<a href="https://colab.research.google.com/github/parthasarathipanda2006/cycle-GAN/blob/main/notebook1c28a56c7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATASET

In [ ]:

import kagglehub
balraj98_monet2photo_path = kagglehub.dataset_download('balraj98/monet2photo')

print('Data source import complete.')


In [ ]:
import tensorflow as tf
import os
from tensorflow.keras import mixed_precision

BUFFER_SIZE = 1000
BATCH_SIZE = 1
IMG_WIDTH = 256
IMG_HEIGHT = 256

def random_crop(image):
    return tf.image.random_crop(image, size=[IMG_HEIGHT, IMG_WIDTH, 3])

def normalize(image):

    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1
    return image

def random_jitter(image):
    image = tf.image.resize(image, [286, 286], method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    image = random_crop(image)
    image = tf.image.random_flip_left_right(image)
    return image

def load_and_decode(file_path):
    image = tf.io.read_file(file_path)
    image = tf.image.decode_jpeg(image, channels=3)
    return image

def augment_and_normalize(image):
    image = random_jitter(image)
    image = normalize(image)
    return image

def create_dataset(folder_path):
    dataset = tf.data.Dataset.list_files(folder_path + '/*.jpg', shuffle=False)
    dataset = dataset.shuffle(BUFFER_SIZE)


    dataset = dataset.map(load_and_decode, num_parallel_calls=tf.data.AUTOTUNE)


    dataset = dataset.cache()


    dataset = dataset.map(augment_and_normalize, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# ==========================================
# KAGGLE DATASET PATHS
# ==========================================
train_A_path = '/kaggle/input/datasets/balraj98/monet2photo/trainB'
train_B_path = '/kaggle/input/datasets/balraj98/monet2photo/trainA'
test_A_path='/kaggle/input/datasets/balraj98/monet2photo/testB'
test_B_path='/kaggle/input/datasets/balraj98/monet2photo/testA'

monet_dataset = create_dataset(train_B_path)
real_dataset = create_dataset(train_A_path)

monet_test = create_dataset(test_B_path)
real_test = create_dataset(test_A_path)

print("Kaggle pipelines built! Ready to train Monet-to-Photo.")

# INSTANCE NORMALIZATION

In [2]:
import tensorflow as tf
from tensorflow.keras import layers

class InstanceNormalization(layers.Layer):
    def __init__(self, epsilon=1e-5, **kwargs):
        super(InstanceNormalization, self).__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):

        self.scale = self.add_weight(
            name='scale',
            shape=(input_shape[-1],),
            initializer=tf.random_normal_initializer(1., 0.02),
            trainable=True
        )

        self.offset = self.add_weight(
            name='offset',
            shape=(input_shape[-1],),
            initializer='zeros',
            trainable=True
        )

    def call(self, x):

        mean, variance = tf.nn.moments(x, axes=[1, 2], keepdims=True)


        inv = tf.math.rsqrt(variance + self.epsilon)
        normalized = (x - mean) * inv

        return self.scale * normalized + self.offset

# GENERATOR

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class ReflectionPadding2D(layers.Layer):
    def __init__(self, padding=(1, 1), **kwargs):
        self.padding = tuple(padding)
        super(ReflectionPadding2D, self).__init__(**kwargs)

    def call(self, input_tensor):
        padding_width, padding_height = self.padding
        return tf.pad(input_tensor,
                      [[0, 0], [padding_height, padding_height], [padding_width, padding_width], [0, 0]],
                      "REFLECT")

def resnet_block(input_layer, filters):
    x = ReflectionPadding2D()(input_layer)
    x = layers.Conv2D(filters, kernel_size=3, padding='valid', use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.Activation('relu')(x)

    x = ReflectionPadding2D()(x)
    x = layers.Conv2D(filters, kernel_size=3, padding='valid', use_bias=False)(x)
    x = InstanceNormalization()(x)

    return layers.Add()([x, input_layer])

def build_generator(name="generator"):
    inputs = layers.Input(shape=[256, 256, 3])

    x = ReflectionPadding2D(padding=(3, 3))(inputs)
    x = layers.Conv2D(64, kernel_size=7, padding='valid', use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(128, kernel_size=3, strides=2, padding='same', use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(256, kernel_size=3, strides=2, padding='same', use_bias=False)(x)
    x = layers.GroupNormalization(groups=-1)(x)
    x = layers.Activation('relu')(x)

    for _ in range(8):
        x = resnet_block(x, 256)

    x = layers.Conv2DTranspose(128, kernel_size=3, strides=2, padding='same', use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2DTranspose(64, kernel_size=3, strides=2, padding='same', use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.Activation('relu')(x)

    x = ReflectionPadding2D(padding=(3, 3))(x)
    outputs = layers.Conv2D(3, kernel_size=7, padding='valid', activation='tanh')(x)

    return Model(inputs, outputs, name=name)

generator_g = build_generator(name="generator_Real_to_Ghibli")
generator_f = build_generator(name="generator_Ghibli_to_Real")

In [4]:

input = tf.random.normal([1, 256, 256, 3])
output = generator_g(input)
output = generator_f(input)
generator_g.summary()

Model: "generator_Real_to_Ghibli"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reflection_padding… │ (None, 262, 262,  │          0 │ input_layer_1[0]… │
│ (ReflectionPadding… │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │      9,408 │ reflection_paddi… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ instance_normaliza… │ (None, 256, 256,  │        128 │ conv2d_1[0][0]    │
│ (InstanceNormaliza… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 256, 256,  │          0 │ instance_normali… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     73,728 │ activation[0][0]  │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ instance_normaliza… │ (None, 128, 128,  │        256 │ conv2d_2[0][0]    │
│ (InstanceNormaliza… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 128, 128,  │          0 │ instance_normali… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │    294,912 │ activation_1[0][… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_normalization │ (None, 64, 64,    │        512 │ conv2d_3[0][0]    │
│ (GroupNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 64,    │          0 │ group_normalizat… │
│ (Activation)        │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reflection_padding… │ (None, 66, 66,    │          0 │ activation_2[0][… │
│ (ReflectionPadding… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │    589,824 │ reflection_paddi… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ instance_normaliza… │ (None, 64, 64,    │        512 │ conv2d_4[0][0]    │
│ (InstanceNormaliza… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 64, 64,    │          0 │ instance_normali… │
│ (Activation)        │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reflection_padding… │ (None, 66, 66,    │          0 │ activation_3[0][… │
│ (ReflectionPadding… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │    589,824 │ reflection_paddi

 Total params: 10,202,755 (38.92 MB)

 Trainable params: 10,202,755 (38.92 MB)

 Non-trainable params: 0 (0.00 B)

#DISCRIMINATOR

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_discriminator(name="discriminator"):
    inputs = layers.Input(shape=[256, 256, 3])

    x = layers.Conv2D(64, kernel_size=4, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2D(128, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.Conv2D(256, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.ZeroPadding2D()(x)
    x = layers.Conv2D(512, kernel_size=4, strides=1, padding='valid', use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)

    x = layers.ZeroPadding2D()(x)
    outputs = layers.Conv2D(1, kernel_size=4, strides=1, padding='valid')(x)

    return Model(inputs, outputs, name=name)

discriminator_x = build_discriminator(name="discriminator_Real")
discriminator_y = build_discriminator(name="discriminator_Ghibli")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [6]:
input=tf.random.normal([1,256,256,3])
output=discriminator_x(input)
output=discriminator_y(input)
discriminator_x.summary()

Model: "discriminator_Real"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_41 (Conv2D)              │ (None, 128, 128, 64)   │         3,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 128, 128, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_42 (Conv2D)              │ (None, 64, 64, 128)    │       131,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ instance_normalization_40       │ (None, 64, 64, 128)    │           256 │
│ (InstanceNormalization)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 64, 64, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_43 (Conv2D)              │ (None, 32, 32, 256)    │       524,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ instance_normalization_41       │ (None, 32, 32, 256)    │           512 │
│ (InstanceNormalization)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 32, 32, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d (ZeroPadding2D)  │ (None, 34, 34, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_44 (Conv2D)              │ (None, 31, 31, 512)    │     2,097,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ instance_normalization_42       │ (None, 31, 31, 512)    │         1,024 │
│ (InstanceNormalization)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_3 (LeakyReLU)       │ (None, 31, 31, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_1                │ (None, 33, 33, 512)    │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_45 (Conv2D)              │ (None, 30, 30, 1)      │         8,193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,765,569 (10.55 MB)

 Trainable params: 2,765,569 (10.55 MB)

 Non-trainable params: 0 (0.00 B)

# LOSSES

In [ ]:
import tensorflow as tf

adv_loss_fn = tf.keras.losses.MeanSquaredError()

def discriminator_loss(real, generated):
    real_loss = adv_loss_fn(tf.ones_like(real), real)
    generated_loss = adv_loss_fn(tf.zeros_like(generated), generated)
    return (real_loss + generated_loss) * 0.5

def generator_loss(generated):
    return adv_loss_fn(tf.ones_like(generated), generated)

def calc_cycle_loss(real_image, cycled_image, lambda_weight=10.0):

    real_image = tf.cast(real_image, tf.float32)
    cycled_image = tf.cast(cycled_image, tf.float32)

    return lambda_weight * tf.reduce_mean(tf.abs(real_image - cycled_image))

def identity_loss(real_image, same_image, lambda_weight=10.0, identity_multiplier=0.3):

    real_image = tf.cast(real_image, tf.float32)
    same_image = tf.cast(same_image, tf.float32)

    return lambda_weight * identity_multiplier * tf.reduce_mean(tf.abs(real_image - same_image))

In [ ]:
class TFImagePool(tf.Module):
    def __init__(self, pool_size=50):
        super(TFImagePool, self).__init__()
        self.pool_size = pool_size

        self.images = tf.Variable(tf.zeros([pool_size, 1, 256, 256, 3], dtype=tf.float32), trainable=False)
        self.count = tf.Variable(0, dtype=tf.int32, trainable=False)

    @tf.function
    def query(self, image):
        if self.pool_size == 0:
            return image


        image_fp32 = tf.cast(image, tf.float32)


        if self.count < self.pool_size:
            self.images[self.count].assign(image_fp32)
            self.count.assign_add(1)
            return image


        else:
            p = tf.random.uniform([])
            if p > 0.5:

                random_id = tf.random.uniform([], 0, self.pool_size, dtype=tf.int32)


                old_image = tf.identity(self.images[random_id])
                old_image = tf.cast(old_image, image.dtype)

                self.images[random_id].assign(image_fp32)
                return old_image
            else:

                return image


fake_x_pool = TFImagePool(pool_size=50)
fake_y_pool = TFImagePool(pool_size=50)

# TRAINING STEP

In [ ]:
import tensorflow as tf

generator_g_optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
generator_f_optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
discriminator_x_optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
discriminator_y_optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)

@tf.function
def train_step(real_x, real_y):
    with tf.GradientTape(persistent=True) as tape:

        fake_y = generator_g(real_x, training=True)
        cycled_x = generator_f(fake_y, training=True)

        fake_x = generator_f(real_y, training=True)
        cycled_y = generator_g(fake_x, training=True)

        same_x = generator_f(real_x, training=True)
        same_y = generator_g(real_y, training=True)


        disc_fake_x_for_gen = discriminator_x(fake_x, training=True)
        disc_fake_y_for_gen = discriminator_y(fake_y, training=True)


        disc_real_x = discriminator_x(real_x, training=True)
        disc_real_y = discriminator_y(real_y, training=True)


        pooled_fake_x = fake_x_pool.query(fake_x)
        pooled_fake_y = fake_y_pool.query(fake_y)


        disc_fake_x_for_disc = discriminator_x(pooled_fake_x, training=True)
        disc_fake_y_for_disc = discriminator_y(pooled_fake_y, training=True)


        gen_g_loss = generator_loss(disc_fake_y_for_gen)
        gen_f_loss = generator_loss(disc_fake_x_for_gen)

        total_cycle_loss = calc_cycle_loss(real_x, cycled_x) + calc_cycle_loss(real_y, cycled_y)

        total_gen_g_loss = gen_g_loss + total_cycle_loss + identity_loss(real_y, same_y)
        total_gen_f_loss = gen_f_loss + total_cycle_loss + identity_loss(real_x, same_x)

        disc_x_loss = discriminator_loss(disc_real_x, disc_fake_x_for_disc)
        disc_y_loss = discriminator_loss(disc_real_y, disc_fake_y_for_disc)


    generator_g_gradients = tape.gradient(total_gen_g_loss, generator_g.trainable_variables)
    generator_f_gradients = tape.gradient(total_gen_f_loss, generator_f.trainable_variables)
    discriminator_x_gradients = tape.gradient(disc_x_loss, discriminator_x.trainable_variables)
    discriminator_y_gradients = tape.gradient(disc_y_loss, discriminator_y.trainable_variables)

    generator_g_optimizer.apply_gradients(zip(generator_g_gradients, generator_g.trainable_variables))
    generator_f_optimizer.apply_gradients(zip(generator_f_gradients, generator_f.trainable_variables))
    discriminator_x_optimizer.apply_gradients(zip(discriminator_x_gradients, discriminator_x.trainable_variables))
    discriminator_y_optimizer.apply_gradients(zip(discriminator_y_gradients, discriminator_y.trainable_variables))

# TRAIN & SAVE

In [ ]:
import matplotlib.pyplot as plt
import os
import time

checkpoint_path = "./checkpoints/train"
ckpt = tf.train.Checkpoint(generator_g=generator_g,
                           generator_f=generator_f,
                           discriminator_x=discriminator_x,
                           discriminator_y=discriminator_y,
                           generator_g_optimizer=generator_g_optimizer,
                           generator_f_optimizer=generator_f_optimizer,
                           discriminator_x_optimizer=discriminator_x_optimizer,
                           discriminator_y_optimizer=discriminator_y_optimizer)

ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=5)

if ckpt_manager.latest_checkpoint:
    ckpt.restore(ckpt_manager.latest_checkpoint)

def generate_images(model, test_input):
    prediction = model(test_input)
    plt.figure(figsize=(12, 12))
    display_list = [test_input[0], prediction[0]]
    title = ['Input Image', 'Generated Ghibli Image']

    for i in range(2):
        plt.subplot(1, 2, i+1)
        plt.title(title[i])

        img_to_plot = tf.cast(display_list[i], tf.float32)

        plt.imshow(img_to_plot * 0.5 + 0.5)
        plt.axis('off')

    plt.show()

def fit(real_dataset, monet_dataset, epochs):

    fixed_input_for_generation = next(iter(real_test))

    steps_per_epoch = 2000
    dataset = tf.data.Dataset.zip((real_dataset, monet_dataset.repeat())).take(2000)
    for epoch in range(epochs):
        start = time.time()
        print(f"\nEpoch {epoch + 1}/{epochs}")



        progbar = tf.keras.utils.Progbar(steps_per_epoch)
        for i, (image_x, image_y) in enumerate(dataset):
            train_step(image_x, image_y)
            progbar.update(i + 1)

        if (epoch + 1) % 2 == 0:
            ckpt_manager.save()
            print("Checkpoint saved")
            generate_images(generator_g, fixed_input_for_generation)

        print(f"Time: {time.time() - start:.2f} sec")
        if (epoch + 1) % 5 == 0:
            generator_g.save(f'generator_g_epoch_{epoch+1}.keras')
            print(f"Saved backup at Epoch {epoch+1}")

EPOCHS = 30
fit(real_dataset, monet_dataset, EPOCHS)